#Upload and read MasterDataset

In [ ]:
from google.colab import files
import pandas as pd
import numpy as np
from scipy.stats import pearsonr

uploaded = files.upload()

filename = list(uploaded.keys())[0]
master = pd.read_excel(filename)

print("Master dataset loaded successfully.")
display(master.head())
print(master.columns)

#Calculate delta scores

In [ ]:
delta_df = pd.DataFrame()

delta_df["ParticipantID"] = master["ParticipantID"]

# PRS delta scores: Smell - NoSmell
delta_df["Delta_TotalPRS"] = master["TotalPRS_Smell"] - master["TotalPRS_NoSmell"]
delta_df["Delta_BeingAway"] = master["BeingAway_Smell"] - master["BeingAway_NoSmell"]
delta_df["Delta_Fascination"] = master["Fascination_Smell"] - master["Fascination_NoSmell"]
delta_df["Delta_Coherence"] = master["Coherence_Smell"] - master["Coherence_NoSmell"]
delta_df["Delta_Scope"] = master["Scope_Smell"] - master["Scope_NoSmell"]

# EEG delta scores: Smell - NoSmell
delta_df["Delta_Attention"] = master["Attention_Task_Smell"] - master["Attention_Task_NoSmell"]
delta_df["Delta_Relaxation"] = master["Relax_Task_Smell"] - master["Relax_Task_NoSmell"]
delta_df["Delta_CognitiveLoad"] = master["CogLoad_Task_Smell"] - master["CogLoad_Task_NoSmell"]

# SART2 delta scores: Smell - NoSmell
delta_df["Delta_CommissionErrors"] = master["CommissionErrors_Smell"] - master["CommissionErrors_NoSmell"]
delta_df["Delta_OmissionErrors"] = master["OmissionErrors_Smell"] - master["OmissionErrors_NoSmell"]
delta_df["Delta_RTVariability"] = master["RTVariability_Smell"] - master["RTVariability_NoSmell"]

print("Delta-score dataset:")
display(delta_df)

#Correlations between PRS and EEG delta scores

In [ ]:
prs_delta_vars = [
    ("Δ Total PRS", "Delta_TotalPRS"),
    ("Δ Being Away", "Delta_BeingAway"),
    ("Δ Fascination", "Delta_Fascination"),
    ("Δ Coherence", "Delta_Coherence"),
    ("Δ Scope", "Delta_Scope")
]

eeg_delta_vars = [
    ("Δ Attention", "Delta_Attention"),
    ("Δ Relaxation", "Delta_Relaxation"),
    ("Δ Cognitive Load", "Delta_CognitiveLoad")
]

prs_eeg_results = []

for prs_label, prs_col in prs_delta_vars:
    for eeg_label, eeg_col in eeg_delta_vars:

        paired_data = delta_df[[prs_col, eeg_col]].dropna()

        if len(paired_data) >= 3:
            r_value, p_value = pearsonr(
                paired_data[prs_col],
                paired_data[eeg_col]
            )

            prs_eeg_results.append({
                "PRS variable": prs_label,
                "EEG variable": eeg_label,
                "r": r_value,
                "p": p_value
            })

        else:
            prs_eeg_results.append({
                "PRS variable": prs_label,
                "EEG variable": eeg_label,
                "r": np.nan,
                "p": np.nan
            })

prs_eeg_df = pd.DataFrame(prs_eeg_results)

display(prs_eeg_df)

prs_eeg_file = "Correlation_PRS_EEG.xlsx" #Table 5.7

prs_eeg_df.to_excel(
    prs_eeg_file,
    index=False
)

print(f"{prs_eeg_file} saved.")

#Correlations between PRS and SART2 delta scores

In [ ]:
sart_delta_vars = [
    ("Δ Commission Errors", "Delta_CommissionErrors"),
    ("Δ Omission Errors", "Delta_OmissionErrors"),
    ("Δ RT Variability", "Delta_RTVariability")
]

prs_sart_results = []

for prs_label, prs_col in prs_delta_vars:
    for sart_label, sart_col in sart_delta_vars:

        paired_data = delta_df[[prs_col, sart_col]].dropna()

        if len(paired_data) >= 3:
            r_value, p_value = pearsonr(
                paired_data[prs_col],
                paired_data[sart_col]
            )

            prs_sart_results.append({
                "PRS variable": prs_label,
                "SART2 variable": sart_label,
                "r": r_value,
                "p": p_value
            })

        else:
            prs_sart_results.append({
                "PRS variable": prs_label,
                "SART2 variable": sart_label,
                "r": np.nan,
                "p": np.nan
            })

prs_sart_df = pd.DataFrame(prs_sart_results)

display(prs_sart_df)

prs_sart_file = "Correlation_PRS_SART2.xlsx" #Table 5.8

prs_sart_df.to_excel(
    prs_sart_file,
    index=False
)

print(f"{prs_sart_file} saved.")

#Correlations between EEG and SART2 delta scores

In [ ]:
eeg_sart_results = []

for eeg_label, eeg_col in eeg_delta_vars:
    for sart_label, sart_col in sart_delta_vars:

        paired_data = delta_df[[eeg_col, sart_col]].dropna()

        if len(paired_data) >= 3:
            r_value, p_value = pearsonr(
                paired_data[eeg_col],
                paired_data[sart_col]
            )

            eeg_sart_results.append({
                "EEG variable": eeg_label,
                "SART2 variable": sart_label,
                "r": r_value,
                "p": p_value
            })

        else:
            eeg_sart_results.append({
                "EEG variable": eeg_label,
                "SART2 variable": sart_label,
                "r": np.nan,
                "p": np.nan
            })

eeg_sart_df = pd.DataFrame(eeg_sart_results)

display(eeg_sart_df)

eeg_sart_file = "Correlation_EEG_SART2.xlsx" #Table 5.9

eeg_sart_df.to_excel(
    eeg_sart_file,
    index=False
)

print(f"{eeg_sart_file} saved.")

#Download results

In [ ]:
from google.colab import files

files.download("DeltaScores_MasterDataset.xlsx")
files.download("Correlation_PRS_EEG.xlsx")    #Table 5.7
files.download("Correlation_PRS_SART2.xlsx")  #Table 5.8
files.download("Correlation_EEG_SART2.xlsx")  #Table 5.9